# Laboratorio L12 - Test di robustezza del filtro antiphishing

B.4 - Cybersicurezza e IA

Il notebook riprende il classificatore di L11 e ne misura la robustezza: si verifica se piccole aggiunte a un messaggio fraudolento, che non ne cambiano la richiesta, bastano a far cambiare la decisione del modello. È il lavoro di chi deve valutare un filtro prima di metterlo in uso.

Tutti i messaggi sono fittizi. Il file `L11_messaggi.csv` deve trovarsi nella stessa cartella del notebook.

## 1. Il filtro di L11

La cella ricostruisce lo stesso filtro di L11 (stessi dati, stessa divisione) e definisce tre funzioni usate nel resto del notebook:

- `addestra(testi, etichette)`: addestra un nuovo filtro
- `prob_phishing(testo, v, m)`: probabilità di phishing di un testo
- `accuratezza(v, m)`: accuratezza sull'insieme di verifica

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

dati = pd.read_csv("L11_messaggi.csv")
X_train, X_test, y_train, y_test = train_test_split(
    dati["testo"], dati["etichetta"], test_size=0.25, random_state=42, stratify=dati["etichetta"])

def addestra(testi, etichette):
    """Addestra un classificatore come in L11 e restituisce (vettorizzatore, modello)."""
    v = CountVectorizer()
    m = MultinomialNB().fit(v.fit_transform(testi), etichette)
    return v, m

def prob_phishing(testo, v, m):
    """Probabilità di phishing assegnata dal modello a un testo."""
    p = m.predict_proba(v.transform([testo]))[0]
    return p[list(m.classes_).index("phishing")]

def accuratezza(v, m):
    return accuracy_score(y_test, m.predict(v.transform(X_test)))

vett, modello = addestra(X_train, y_train)
print("accuratezza del filtro di L11:", round(accuratezza(vett, modello), 3))

## 2. Le parole che spingono verso "legittimo" (esercizio 2)

Come in L11, per ogni parola si calcola quanto è più frequente in una classe che nell'altra. Qui interessano le parole all'estremo "legittimo": sono le leve che un attaccante può usare.

In [ ]:
parole = vett.get_feature_names_out()
classi = list(modello.classes_)
diff = modello.feature_log_prob_[classi.index("phishing")] - modello.feature_log_prob_[classi.index("legittimo")]
parole_legittime = list(parole[diff.argsort()[:40]])   # le 40 più indicative di "legittimo"
print(", ".join(parole_legittime[:15]))

**Domanda**: queste parole descrivono che cosa rende legittimo un messaggio, o soltanto che cosa c'era nei messaggi legittimi del dataset?

Risposta:

## 3. Aggiunte manuali (esercizio 3)

Al messaggio fraudolento si aggiungono alcune parole in coda, senza cambiare la richiesta. In un'email reale il testo aggiunto potrebbe essere reso poco visibile (in fondo, in caratteri piccoli o dello stesso colore dello sfondo): il filtro lo legge, la persona no.

Modificare le variabili `aggiunta_breve` e `aggiunta_lunga` e osservare la probabilità. Cercare l'aggiunta più breve che porta la probabilità sotto 0,5.

In [ ]:
originale = "Gentile cliente, SpediVeloce: il tuo pacco è in giacenza, paga le spese di 1,99 euro entro 24 ore al link"
aggiunta_breve = "gentili famiglie registro"
aggiunta_lunga = "buongiorno gentili famiglie, circolare nel registro elettronico, lezioni regolarmente"

print(f"originale                 : {prob_phishing(originale, vett, modello):.3f}")
print(f"originale + aggiunta breve: {prob_phishing(originale + ' ' + aggiunta_breve, vett, modello):.3f}")
print(f"originale + aggiunta lunga: {prob_phishing(originale + ' ' + aggiunta_lunga, vett, modello):.3f}")

## 4. Ricerca automatica (esercizio 4)

La funzione `aggiungi_parole` automatizza il procedimento: a ogni passo prova le 40 parole più "legittime", aggiunge quella che abbassa di più la probabilità di phishing e si ferma quando la probabilità scende sotto 0,5 (o dopo 30 parole).

La funzione è applicata a tutti i messaggi fraudolenti dell'insieme di verifica che il filtro riconosce. Il risultato è una misura di robustezza: quanti messaggi si possono far passare e con quante parole.

In [ ]:
def aggiungi_parole(testo, v, m, candidate, max_parole=30):
    aggiunte = []
    while prob_phishing(testo, v, m) >= 0.5 and len(aggiunte) < max_parole:
        migliore = min(candidate, key=lambda w: prob_phishing(testo + " " + w, v, m))
        testo = testo + " " + migliore
        aggiunte.append(migliore)
    return testo, aggiunte

riconosciuti = [t for t in X_test[y_test == "phishing"] if prob_phishing(t, vett, modello) >= 0.5]
risultati = [aggiungi_parole(t, vett, modello, parole_legittime) for t in riconosciuti]

passati = [agg for testo, agg in risultati if prob_phishing(testo, vett, modello) < 0.5]
print("messaggi fraudolenti riconosciuti dal filtro:", len(riconosciuti))
print("messaggi che passano dopo le aggiunte     :", len(passati))
print("parole aggiunte in media                  :", round(np.mean([len(a) for a in passati]), 1))
print()
testo, agg = risultati[0]
print("esempio:", testo)

**Domande**

- La richiesta del messaggio è cambiata?
- Perché il procedimento funziona con un modello "a sacchetto di parole"?
- Una persona che legge il messaggio e applica la verifica indipendente sarebbe ingannata?

Risposte:

## 5. Una difesa e la sua risposta (esercizio 5, approfondimento)

**Addestramento con esempi avversari**: il difensore genera messaggi fraudolenti con parole aggiunte, li etichetta come phishing e li inserisce nei dati di addestramento.

La cella verifica due cose:

- il nuovo filtro riconosce i messaggi modificati del punto 4?
- chi conosce il nuovo filtro può ripetere la ricerca e far passare di nuovo i messaggi?

In [ ]:
esempi_avversari = [aggiungi_parole(t, vett, modello, parole_legittime)[0] for t in X_train[y_train == "phishing"]]
vett2, modello2 = addestra(pd.concat([X_train, pd.Series(esempi_avversari)]),
                           pd.concat([y_train, pd.Series(["phishing"] * len(esempi_avversari))]))
print("accuratezza del nuovo filtro:", round(accuratezza(vett2, modello2), 3))

vecchi = [testo for testo, agg in risultati]
print("messaggi modificati del punto 4 riconosciuti:", sum(prob_phishing(t, vett2, modello2) >= 0.5 for t in vecchi), "su", len(vecchi))

# l'attaccante ripete la ricerca sul nuovo filtro
parole2 = vett2.get_feature_names_out()
diff2 = modello2.feature_log_prob_[list(modello2.classes_).index("phishing")] - modello2.feature_log_prob_[list(modello2.classes_).index("legittimo")]
candidate2 = list(parole2[diff2.argsort()[:40]])
nuovi = [aggiungi_parole(t, vett2, modello2, candidate2) for t in riconosciuti]
print("messaggi che passano dopo una nuova ricerca :", sum(prob_phishing(t, vett2, modello2) < 0.5 for t, a in nuovi), "su", len(nuovi))

**Domande**

- L'addestramento con esempi avversari ha reso il filtro sicuro?
- Quali difese non dipendono dalle parole del messaggio (L2, L8)?

Risposte: